# Phase 9 — LoRA fine-tuning of Qwen2.5-Coder-1.5B-Instruct on COBOL Business-Intent Cards

**Goal:** Adapt the small Qwen-Coder model so it produces faithful, JSON-valid Business Intent Cards for unseen COBOL blocks — outperforming the base model on the Phase 8 ablation table.

**Inputs from the repo:**
- `outputs/finetune_train.jsonl` — chat-format SFT corpus (built by `python -m cobol_archaeologist.cli finetune-build`).

**Output of this notebook:**
- A merged-and-quantized GGUF model + Modelfile that you can load into Ollama as `cobol-archaeologist:v1`.

**Runtime:** Free Colab T4 (16 GB) is enough for Qwen-1.5B + LoRA in 4-bit. ~10-20 min for the included corpus.

## 0. Install dependencies

In [ ]:
!pip -q install \
    transformers==4.46.3 \
    peft==0.13.2 \
    trl==0.12.1 \
    accelerate==1.1.1 \
    bitsandbytes==0.44.1 \
    datasets==3.1.0

## 1. Point to training JSONL

`TRAIN_FILE` is set to `../outputs/finetune_train.jsonl` (relative to this notebook). Change it if your file is elsewhere.

In [ ]:
# Local path to training data (no Colab upload needed)
TRAIN_FILE = "../outputs/finetune_train.jsonl"

In [ ]:
from datasets import load_dataset

ds = load_dataset("json", data_files=TRAIN_FILE, split="train")
print(ds)
print(ds[0]["messages"][1]["content"][:300])

## 2. Load Qwen2.5-Coder-1.5B-Instruct in 4-bit

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-Coder-1.5B-Instruct"

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb,
    device_map="auto",
)
model.config.use_cache = False

## 3. Configure LoRA

In [ ]:
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model

model = prepare_model_for_kbit_training(model)

lora_cfg = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

## 4. Convert messages -> chat-templated text

TRL's `SFTTrainer` will call `tokenizer.apply_chat_template` automatically when given a `messages` field, but we keep the conversion explicit for transparency.

In [ ]:
def to_text(example):
    text = tokenizer.apply_chat_template(
        example["messages"], tokenize=False, add_generation_prompt=False
    )
    return {"text": text}

ds_text = ds.map(to_text, remove_columns=ds.column_names)
print(ds_text[0]["text"][:600])

## 5. Train

In [ ]:
from trl import SFTConfig, SFTTrainer

cfg = SFTConfig(
    output_dir="./cobol-archaeologist-lora",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    logging_steps=2,
    save_strategy="epoch",
    bf16=False,
    fp16=True,
    max_seq_length=2048,
    packing=False,
    dataset_text_field="text",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=cfg,
    train_dataset=ds_text,
    tokenizer=tokenizer,
)
trainer.train()

## 6. Merge LoRA into the base weights

In [ ]:
from peft import PeftModel

trainer.model.save_pretrained("cobol-archaeologist-lora/adapter")
tokenizer.save_pretrained("cobol-archaeologist-lora/adapter")

# Reload base in fp16 (no 4-bit) so we can merge cleanly.
base = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map="auto")
merged = PeftModel.from_pretrained(base, "cobol-archaeologist-lora/adapter")
merged = merged.merge_and_unload()
merged.save_pretrained("cobol-archaeologist-merged", safe_serialization=True)
tokenizer.save_pretrained("cobol-archaeologist-merged")

## 7. Quick sanity generation

In [ ]:
msgs = ds[0]["messages"][:2]  # system + user, no assistant
prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
inp = tokenizer(prompt, return_tensors="pt").to(merged.device)
out = merged.generate(**inp, max_new_tokens=300, do_sample=False)
print(tokenizer.decode(out[0][inp.input_ids.shape[1]:], skip_special_tokens=True))

## 8. Convert to GGUF + load into Ollama (run on a machine with `llama.cpp`)

Download `cobol-archaeologist-merged/` from Colab. Then locally:

```bash
git clone https://github.com/ggerganov/llama.cpp
cd llama.cpp
pip install -r requirements.txt
python convert_hf_to_gguf.py ../cobol-archaeologist-merged \
    --outfile ../cobol-archaeologist.f16.gguf --outtype f16
./build/bin/llama-quantize ../cobol-archaeologist.f16.gguf \
    ../cobol-archaeologist.q4_k_m.gguf q4_k_m
```

Then create a `Modelfile` (also stored in this repo as `scripts/Modelfile.cobol-archaeologist`) and load it:

```bash
ollama create cobol-archaeologist:v1 -f scripts/Modelfile.cobol-archaeologist
```

Now back in the project repo:

```powershell
python -m cobol_archaeologist.cli compare-baselines \
    --golden data/generated/golden_eval.jsonl \
    --backend ollama --backend-args '{"model":"cobol-archaeologist:v1"}' \
    --index-dir data/index --offline --out-dir reports
```

Compare the new row against the Phase-8 baseline in `reports/comparison_table.md`.